In [0]:
from pyspark.sql.functions import col, min, datediff, count, avg, when

maint = spark.table("workspace.default.silver_maint")
failures = spark.table("workspace.default.silver_failures")

maint.printSchema()
failures.printSchema()

root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- comp: string (nullable = true)

root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- failure: string (nullable = true)



In [0]:
maintenance_failure = (
    maint.alias("m")
    .join(
        failures.alias("f"),
        (col("m.machineID") == col("f.machineID")) &
        (col("m.comp") == col("f.failure")) &
        (col("f.datetime") > col("m.datetime")),
        "left"
    )
    .groupBy(
        col("m.datetime").alias("maintenance_datetime"),
        col("m.machineID").alias("machineID"),
        col("m.comp").alias("component")
    )
    .agg(
        min("f.datetime").alias("next_failure_datetime")
    )
)

print("Maintenance events matched:", maintenance_failure.count())

display(maintenance_failure.limit(10))

Maintenance events matched: 3286


maintenance_datetime,machineID,component,next_failure_datetime
2015-02-04T06:00:00.000Z,1,comp4,2015-06-19T06:00:00.000Z
2014-11-28T06:00:00.000Z,3,comp2,2015-01-07T06:00:00.000Z
2015-07-01T06:00:00.000Z,4,comp1,2015-08-30T06:00:00.000Z
2015-01-19T06:00:00.000Z,10,comp2,2015-04-04T06:00:00.000Z
2015-02-19T06:00:00.000Z,11,comp3,2015-04-20T06:00:00.000Z
2015-07-10T06:00:00.000Z,13,comp2,2015-08-09T06:00:00.000Z
2014-10-14T06:00:00.000Z,14,comp2,2015-01-31T06:00:00.000Z
2015-05-31T06:00:00.000Z,14,comp1,2015-07-15T06:00:00.000Z
2014-06-01T06:00:00.000Z,21,comp3,2015-01-23T06:00:00.000Z
2015-07-07T06:00:00.000Z,21,comp2,2015-08-06T06:00:00.000Z


In [0]:
maintenance_failure = maintenance_failure.withColumn(
    "days_to_failure",
    datediff(
        col("next_failure_datetime"),
        col("maintenance_datetime")
    )
)

display(
    maintenance_failure
    .select(
        "machineID",
        "component",
        "maintenance_datetime",
        "next_failure_datetime",
        "days_to_failure"
    )
    .limit(10)
)

machineID,component,maintenance_datetime,next_failure_datetime,days_to_failure
1,comp4,2015-02-04T06:00:00.000Z,2015-06-19T06:00:00.000Z,135
3,comp2,2014-11-28T06:00:00.000Z,2015-01-07T06:00:00.000Z,40
4,comp1,2015-07-01T06:00:00.000Z,2015-08-30T06:00:00.000Z,60
10,comp2,2015-01-19T06:00:00.000Z,2015-04-04T06:00:00.000Z,75
11,comp3,2015-02-19T06:00:00.000Z,2015-04-20T06:00:00.000Z,60
13,comp2,2015-07-10T06:00:00.000Z,2015-08-09T06:00:00.000Z,30
14,comp2,2014-10-14T06:00:00.000Z,2015-01-31T06:00:00.000Z,109
14,comp1,2015-05-31T06:00:00.000Z,2015-07-15T06:00:00.000Z,45
21,comp3,2014-06-01T06:00:00.000Z,2015-01-23T06:00:00.000Z,236
21,comp2,2015-07-07T06:00:00.000Z,2015-08-06T06:00:00.000Z,30


In [0]:
maintenance_failure_status = (
    maintenance_failure
    .withColumn(
        "maintenance_status",
        when(
            col("next_failure_datetime").isNull(),
            "No Failure After Maintenance"
        ).otherwise("Failure After Maintenance")
    )
)

display(
    maintenance_failure_status
    .groupBy("maintenance_status")
    .count()
)

maintenance_status,count
Failure After Maintenance,1432
No Failure After Maintenance,1854


In [0]:
status_summary = (
    maintenance_failure_status
    .groupBy("maintenance_status")
    .count()
)

total_events = maintenance_failure_status.count()

display(
    status_summary
    .withColumn(
        "percentage",
        (col("count") / total_events * 100)
    )
)

maintenance_status,count,percentage
Failure After Maintenance,1432,43.57881923311017
No Failure After Maintenance,1854,56.42118076688983


In [0]:
failure_timing = (
    maintenance_failure
    .filter(col("days_to_failure").isNotNull())
    .agg(
        count("*").alias("failure_events"),
        avg("days_to_failure").alias("avg_days_to_failure"),
        min("days_to_failure").alias("min_days_to_failure")
    )
)

display(failure_timing)

failure_events,avg_days_to_failure,min_days_to_failure
1432,113.24371508379889,1


In [0]:
failure_timing_groups = (
    maintenance_failure
    .filter(col("days_to_failure").isNotNull())
    .withColumn(
        "timing_group",
        when(col("days_to_failure") <= 30, "0-30 days")
        .when(col("days_to_failure") <= 90, "31-90 days")
        .when(col("days_to_failure") <= 180, "91-180 days")
        .otherwise("181+ days")
    )
    .groupBy("timing_group")
    .count()
    .orderBy("timing_group")
)

display(failure_timing_groups)

timing_group,count
0-30 days,87
181+ days,221
31-90 days,694
91-180 days,430


In [0]:
component_summary = (
    maintenance_failure_status
    .groupBy("component", "maintenance_status")
    .count()
    .orderBy("component", "maintenance_status")
)

display(component_summary)

component,maintenance_status,count
comp1,Failure After Maintenance,398
comp1,No Failure After Maintenance,406
comp2,Failure After Maintenance,567
comp2,No Failure After Maintenance,296
comp3,Failure After Maintenance,183
comp3,No Failure After Maintenance,625
comp4,Failure After Maintenance,284
comp4,No Failure After Maintenance,527


In [0]:
gold_q9 = (
    maintenance_failure_status
    .select(
        "machineID",
        "component",
        "maintenance_datetime",
        "next_failure_datetime",
        "days_to_failure",
        "maintenance_status"
    )
)

gold_q9.printSchema()

root
 |-- machineID: integer (nullable = true)
 |-- component: string (nullable = true)
 |-- maintenance_datetime: timestamp (nullable = true)
 |-- next_failure_datetime: timestamp (nullable = true)
 |-- days_to_failure: integer (nullable = true)
 |-- maintenance_status: string (nullable = false)



In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.gold_q9_maintenance_effectiveness"
)

target.alias("t").merge(
    gold_q9.alias("s"),
    """
    t.machineID = s.machineID
    AND t.component = s.component
    AND t.maintenance_datetime = s.maintenance_datetime
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Q9 table updated using MERGE")

spark.table(
    "workspace.default.gold_q9_maintenance_effectiveness"
).printSchema()

print(
    "Gold Q9 row count:",
    spark.table(
        "workspace.default.gold_q9_maintenance_effectiveness"
    ).count()
)

Gold Q9 table updated using MERGE
root
 |-- machineID: integer (nullable = true)
 |-- component: string (nullable = true)
 |-- maintenance_datetime: timestamp (nullable = true)
 |-- next_failure_datetime: timestamp (nullable = true)
 |-- days_to_failure: integer (nullable = true)
 |-- maintenance_status: string (nullable = true)

Gold Q9 row count: 3286


In [0]:
display(
    spark.table(
        "workspace.default.gold_q9_maintenance_effectiveness"
    )
    .groupBy("maintenance_status")
    .count()
)

maintenance_status,count
Failure After Maintenance,1432
No Failure After Maintenance,1854


Databricks visualization. Run in Databricks to view.